Segmentación de Clientes y Predicción de Compra
Contexto

Eres analista de datos en una empresa de comercio electrónico que quiere mejorar su estrategia de marketing mediante la segmentación de clientes y la predicción de su intención de compra.

Tu objetivo es crear un sistema que:

Genere datos sintéticos representativos de clientes reales, con variables como cuánto gastan, cuántas compras hacen, y con qué frecuencia compran.

Segmente a los clientes en grupos similares usando un algoritmo de clustering.

Entrene un modelo predictivo para estimar si un cliente comprará en el próximo mes basándose en sus características y el segmento al que pertenece.

Visualice los segmentos y la probabilidad de compra para facilitar la interpretación de los resultados.



Datos proporcionados y estructura

Clase CustomerDataGenerator

Esta clase debe generar un DataFrame con 300 clientes sintéticos, cada uno con estas columnas:

total_spent: Dinero total gastado por el cliente, en euros (valor aleatorio entre 50 y 1500).

total_purchases: Número total de compras realizadas (entero entre 1 y 50).

purchase_frequency: Frecuencia de compra mensual (valor entre 0.5 y 10).

will_buy_next_month: Etiqueta binaria (1 o 0) que indica si el cliente comprará el próximo mes. La regla para asignar 1 es: si total_spent > 500 y purchase_frequency > 4, el cliente comprará (1), si no, no comprará (0).



Modelado

Clase CustomerSegmentationModel

Esta clase debe:

Recibir el DataFrame generado.

Segmentar clientes en 3 grupos usando KMeans con las variables total_spent, total_purchases y purchase_frequency.

Añadir la columna customer_segment al DataFrame con el número de segmento asignado a cada cliente.

Entrenar un modelo de regresión logística para predecir will_buy_next_month, usando como variables las originales más la segmentación (transformada en variables dummy).

Proveer métodos para obtener la precisión del modelo y la matriz de confusión.



Visualizaciones

Función graficar_segmentos(data):

Genera un scatter plot de total_spent vs purchase_frequency.

Usa colores diferentes para cada segmento.

Añade leyenda, etiquetas y título descriptivo.

Función graficar_probabilidad_compra(modelo):

Muestra cómo varía la probabilidad de compra del cliente en función del gasto total (total_spent), manteniendo constantes total_purchases=25 y purchase_frequency=5.

Dibuja la curva de probabilidad predicha por el modelo de regresión logística.



Indicaciones numéricas y técnicas

Número de muestras: 300.

Número de clusters para KMeans: 3.

Random seed: 42 para reproducibilidad.

División de datos para entrenamiento/prueba: 80% / 20%.

Iteraciones máximas para la regresión logística: 500.

Uso solo de numpy, pandas, sklearn y matplotlib



Ejemplo de uso

# 1. Generar datos

generador = CustomerDataGenerator()

datos_clientes = generador.generate(300)
 
# 2. Crear modelo

modelo = CustomerSegmentationModel(datos_clientes)

modelo.segment_customers()

modelo.train_model()
 
# 3. Resultados

print("Precisión del modelo:", modelo.get_accuracy())

print("Matriz de confusión:\n", modelo.get_confusion_matrix())
 
# 4. Visualizaciones

graficar_segmentos(modelo.data)

graficar_probabilidad_compra(modelo.model)


Salida esperada

Precisión del modelo: 0.8833333333333333

Matriz de confusión:

 [[30  2]
 
 [ 5 23]]









Practicar con diferentes algoritmos de forma conjunta


# Solucion:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# --- 1. Generación de Datos ---
class CustomerDataGenerator:
    def generate(self, n_samples=300):
        np.random.seed(42)
        
        # Generar variables independientes
        total_spent = np.random.uniform(50, 1500, n_samples)
        total_purchases = np.random.randint(1, 51, n_samples)
        purchase_frequency = np.random.uniform(0.5, 10, n_samples)
        
        # Generar target (will_buy_next_month)
        will_buy = []
        for spent, freq in zip(total_spent, purchase_frequency):
            if spent > 500 and freq > 4:
                will_buy.append(1)
            else:
                will_buy.append(0)
                
        # Crear DataFrame
        df = pd.DataFrame({
            'total_spent': total_spent,
            'total_purchases': total_purchases,
            'purchase_frequency': purchase_frequency,
            'will_buy_next_month': will_buy
        })
        return df

# --- 2. Modelo de Segmentación y Predicción (CORREGIDO) ---
class CustomerSegmentationModel:
    def __init__(self, data):
        self.data = data.copy()
        self.kmeans = None
        self.model = None
        self.X_test = None
        self.y_test = None
        self.feature_names = None
        
    # CORRECCIÓN AQUÍ: Añadimos el parámetro n_clusters con valor por defecto 3
    def segment_customers(self, n_clusters=3):
        # Usamos las 3 variables numéricas para el clustering
        features = self.data[['total_spent', 'total_purchases', 'purchase_frequency']]
        
        # Usamos el n_clusters que nos pasen (o 3 por defecto)
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.data['customer_segment'] = self.kmeans.fit_predict(features)
        
    def train_model(self):
        # 1. Preparar datos: One-Hot Encoding de los segmentos
        data_with_dummies = pd.get_dummies(self.data, columns=['customer_segment'], prefix='segment')
        
        # Definir Features (X) y Target (y)
        X = data_with_dummies.drop(columns=['will_buy_next_month'])
        y = data_with_dummies['will_buy_next_month']
        
        # 2. Split train/test (80/20)
        X_train, self.X_test, y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        # 3. Entrenar Regresión Logística
        self.model = LogisticRegression(max_iter=500, random_state=42)
        self.model.fit(X_train, y_train)
        
        # Guardamos nombres de columnas para usarlos luego
        self.feature_names = X.columns.tolist()

    def get_accuracy(self):
        predicciones = self.model.predict(self.X_test)
        return accuracy_score(self.y_test, predicciones)
    
    def get_confusion_matrix(self):
        predicciones = self.model.predict(self.X_test)
        return confusion_matrix(self.y_test, predicciones)

# --- 3. Funciones de Visualización ---

def graficar_segmentos(data):
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        data['total_spent'], 
        data['purchase_frequency'], 
        c=data['customer_segment'], 
        cmap='viridis', 
        alpha=0.7
    )
    plt.colorbar(scatter, label='Segmento')
    plt.xlabel('Gasto Total (€)')
    plt.ylabel('Frecuencia')
    plt.title('Segmentación de Clientes')
    plt.show()

def graficar_probabilidad_compra(model_wrapper):
    plt.figure(figsize=(10, 6))
    rango_gasto = np.linspace(50, 1500, 100)
    
    # DataFrame base
    df_pred = pd.DataFrame({
        'total_spent': rango_gasto,
        'total_purchases': 25,
        'purchase_frequency': 5
    })
    
    # Rellenar dummies (Asumiendo segmento 0 para la visualización)
    # Si el modelo no se ha entrenado, feature_names podría no existir, pero el test asume que sí.
    if hasattr(model_wrapper, 'feature_names') and model_wrapper.feature_names:
        for col in model_wrapper.feature_names:
            if col not in df_pred.columns:
                df_pred[col] = 1 if 'segment_0' in col else 0
        
        # Asegurar orden de columnas
        df_pred = df_pred[model_wrapper.feature_names]
        
        probs = model_wrapper.model.predict_proba(df_pred)[:, 1]
        
        plt.plot(rango_gasto, probs, color='blue')
        plt.title('Probabilidad de Compra (Frecuencia=5, Segmento=0)')
        plt.xlabel('Gasto Total')
        plt.ylabel('Probabilidad')
        plt.ylim(-0.1, 1.1)
        plt.show()

Casos de prueba

Suspenso: 0, Aprobado: 4 de 4 pruebas

test_confusion_matrix_shape

test_generar_datos

test_segment_customers

test_train_model